In [1]:
import pandas as pd

df = pd.read_parquet("labelled_v2.parquet")
df["eventTime"] = pd.to_datetime(df["eventTime"], errors="coerce")
inc = pd.read_parquet("incidents_v2.parquet")

SHOW_COLS = [c for c in ["eventTime", "eventName", "eventSource", "sourceIPAddress",
                         "userIdentityarn", "errorCode", "userAgent"] if c in df.columns]
KEY_COLS = [c for c in ["eventName", "eventSource", "sourceIPAddress", "userIdentityarn", "errorCode"] if c in df.columns]

real = inc[~inc["is_routine_noise"]]
ex1 = real[real["label_category"] == "credential_abuse"].sort_values("n_events", ascending=False).iloc[0]
owner = real[real["label_category"] == "benign_owner_activity"].sort_values("n_events")
ex2 = owner.iloc[len(owner) // 2]
print("Loaded", len(df), "events and", len(inc), "incidents")

Loaded 1225784 events and 4652 incidents


In [2]:
import re

ACCOUNT_ID = "811596193553"
_role, _name, _sess = {}, {}, {}

def anon_arn(a):
    a = str(a)
    if a == "None":
        return a
    out = a.replace(ACCOUNT_ID, "123456789012")
    m = re.match(r"^(.*:assumed-role/)([^/]+)/(.+)$", out)
    if m:
        role = _role.setdefault(m.group(2), f"role-{len(_role) + 1}")
        s = m.group(3)
        sess = _sess.setdefault(s, ("ec2-instance-" if s.startswith("i-") else "session-") + str(len(_sess) + 1))
        return m.group(1) + role + "/" + sess
    m = re.match(r"^(.*:(?:user|role)/)(.+)$", out)
    if m:
        return m.group(1) + _name.setdefault(m.group(2), f"name-{len(_name) + 1}")
    return out

if "arn_real" not in df.columns:          # protects against running this cell twice
    ARN_MAP = {a: anon_arn(a) for a in sorted(df["userIdentityarn"].astype(str).unique())}
    df["arn_real"] = df["userIdentityarn"].astype(str)
    df["userIdentityarn"] = df["arn_real"].map(ARN_MAP)
    inc["arn_real"] = inc["userIdentityarn"].astype(str)
    inc["userIdentityarn"] = inc["arn_real"].map(ARN_MAP)

real = inc[~inc["is_routine_noise"]]
ex1 = real[real["label_category"] == "credential_abuse"].sort_values("n_events", ascending=False).iloc[0]
owner = real[real["label_category"] == "benign_owner_activity"].sort_values("n_events")
ex2 = owner.iloc[len(owner) // 2]

print(f"Hidden: {len(_role)} role names, {len(_name)} user names, {len(_sess)} session names")
print("ex1 is now:", ex1["userIdentityarn"])
print("ex2 is now:", ex2["userIdentityarn"])

Hidden: 16 role names, 5 user names, 49 session names
ex1 is now: arn:aws:sts::123456789012:assumed-role/role-12/ec2-instance-41
ex2 is now: arn:aws:iam::123456789012:root


In [3]:
def collapse(rows):
    r = rows[SHOW_COLS].copy()
    if "userAgent" in r.columns:
        r["userAgent"] = r["userAgent"].astype(str).str[:40]
    keys = r[KEY_COLS].astype(str)
    block = (keys != keys.shift()).any(axis=1).cumsum()
    agg = {c: (c, "first") for c in SHOW_COLS}
    agg["count"] = ("eventName", "size")
    return r.groupby(block).agg(**agg).reset_index(drop=True)

def head_tail(t, n):
    return t if len(t) <= n else pd.concat([t.head(n // 2), t.tail(n - n // 2)])

def list_counts(s):
    return ", ".join(f"{k} x{v}" for k, v in s.items()) or "none"

def get_context(incident, df, window_hours=1, core_rows=20, around_rows=10):
    h, arn, ip = incident["hour"], incident["userIdentityarn"], incident["sourceIPAddress"]
    start, end = h - pd.Timedelta(hours=window_hours), h + pd.Timedelta(hours=1 + window_hours)
    in_time = (df["eventTime"] >= start) & (df["eventTime"] < end)
    ctx = df.loc[in_time & ((df["userIdentityarn"] == arn) | (df["sourceIPAddress"] == ip))].sort_values("eventTime")

    in_hour = (ctx["eventTime"] >= h) & (ctx["eventTime"] < h + pd.Timedelta(hours=1))
    core = ctx[in_hour & (ctx["userIdentityarn"] == arn) & (ctx["sourceIPAddress"] == ip)]
    around = ctx.drop(core.index)

    other_ids = ctx.loc[ctx["sourceIPAddress"] == ip, "userIdentityarn"].value_counts().drop(arn, errors="ignore").head(5)
    other_ips = ctx.loc[ctx["userIdentityarn"] == arn, "sourceIPAddress"].value_counts().drop(ip, errors="ignore").head(5)

    lines = [
        f"INCIDENT: identity={arn} | ip={ip} | hour={h} | identity type={incident['identity_type']} | flagged events={incident['n_events']}",
        f"CONTEXT WINDOW: {start} to {end} | related events={len(ctx)}",
        "TOP EVENTS IN WINDOW: " + list_counts(ctx["eventName"].value_counts().head(10)),
        "OTHER IDENTITIES USING THIS IP: " + list_counts(other_ids),
        "OTHER IPs USED BY THIS IDENTITY: " + list_counts(other_ips),
    ]
    if "errorCode" in ctx.columns:
        lines.append("ERRORS: " + list_counts(ctx.loc[ctx["errorCode"] != "None", "errorCode"].value_counts().head(5)))
    if len(core):
        core_c = head_tail(collapse(core), core_rows)
        lines.append(f"\nINCIDENT-HOUR EVENTS ({len(core)} events, {len(core_c)} rows shown, repeats merged):")
        lines.append(core_c.to_string(index=False))
    if len(around):
        ar_c = head_tail(collapse(around), around_rows)
        lines.append(f"\nSURROUNDING EVENTS ({len(around)} events, {len(ar_c)} rows shown, repeats merged):")
        lines.append(ar_c.to_string(index=False))
    return "\n".join(lines)

print("get_context ready")

get_context ready


In [4]:
def history_lines(incident, df):
    arn, ip, h = incident["userIdentityarn"], incident["sourceIPAddress"], incident["hour"]
    same_id = df["userIdentityarn"] == arn
    same_ip = df["sourceIPAddress"] == ip
    before = df["eventTime"] < h

    id_prior = df.loc[same_id & before, "eventTime"]
    pair_prior = df.loc[same_id & same_ip & before, "eventTime"]
    id_txt = (f"this identity was first seen {id_prior.min():%Y-%m-%d} and was active on {id_prior.dt.date.nunique()} earlier days"
              if len(id_prior) else "this identity has NOT been seen before (first appearance)")
    pair_txt = (f"it used this IP on {pair_prior.dt.date.nunique()} earlier days (first {pair_prior.min():%Y-%m-%d})"
                if len(pair_prior) else "it has NEVER used this IP before")

    in_hour = (df["eventTime"] >= h) & (df["eventTime"] < h + pd.Timedelta(hours=1))
    ua = df.loc[same_id & same_ip & in_hour, "userAgent"].astype(str).str.lower()
    console = ua.str.contains("console")
    script = ~console & ua.str.contains("boto|aws-cli|aws-sdk|botocore|python|terraform|curl")
    internal = ~console & ~script & ua.str.contains("aws-internal|amazonaws.com|awscloudtrail")
    other = ~(console | script | internal)
    return [
        f"HISTORY: {id_txt}; {pair_txt}.",
        f"ACCESS METHOD (incident hour): web console x{int(console.sum())}, CLI/SDK script x{int(script.sum())}, "
        f"AWS internal service x{int(internal.sum())}, other x{int(other.sum())}",
    ]

def build_context(incident, df):
    lines = get_context(incident, df).split("\n")
    return "\n".join(lines[:1] + history_lines(incident, df) + lines[1:])

for ex in [ex1, ex2]:
    text = build_context(ex, df)
    print("\n".join(text.split("\n")[:8]))
    leaks = [w for w in ["flaws", "level5", "level6", "811596193553"] if w in text.lower()]
    print("LEAK CHECK:", "OK, nothing found" if not leaks else f"FOUND {leaks}")
    print("=" * 80)

INCIDENT: identity=arn:aws:sts::123456789012:assumed-role/role-12/ec2-instance-41 | ip=251.105.254.1 | hour=2020-06-11 21:00:00+00:00 | identity type=AssumedRole | flagged events=428
HISTORY: this identity was first seen 2017-02-28 and was active on 421 earlier days; it used this IP on 2 earlier days (first 2020-06-10).
ACCESS METHOD (incident hour): web console x0, CLI/SDK script x3336, AWS internal service x0, other x0
CONTEXT WINDOW: 2020-06-11 20:00:00+00:00 to 2020-06-11 23:00:00+00:00 | related events=3463
TOP EVENTS IN WINDOW: DescribeEventCategories x45, DescribeEventSubscriptions x40, DescribeDBInstances x36, DescribeDBEngineVersions x35, DescribeDBSubnetGroups x35, DescribeDBParameterGroups x30, DescribeDBSecurityGroups x26, DescribeDBSnapshots x26, DescribeOptionGroups x25, DescribeReservedDBInstancesOfferings x25
OTHER IDENTITIES USING THIS IP: arn:aws:iam::123456789012:user/name-4 x4
OTHER IPs USED BY THIS IDENTITY: apigateway.amazonaws.com x120
ERRORS: AccessDenied x2974,

In [5]:
import os, time, getpass
from google import genai

if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass.getpass("Paste your Gemini API key: ")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
print("Gemini connected")

Gemini connected


In [6]:
import json
from pydantic import BaseModel
from typing import Literal, List

MODEL = "gemini-3.5-flash-lite"
CACHE_FILE = "llm_cache.json"
PAUSE_SECONDS = 7          # wait between calls to respect the per-minute limit

try:
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        CACHE = json.load(f)
except FileNotFoundError:
    CACHE = {}

class RCAResult(BaseModel):
    verdict: Literal["malicious", "suspicious", "benign"]
    root_cause: str
    severity: Literal["low", "medium", "high", "critical"]
    confidence: float
    evidence: List[str]
    next_action: str

INSTRUCTIONS = """You are a cloud security analyst reviewing AWS CloudTrail activity.
An anomaly detector flagged the incident below. Anomaly detectors flag rare activity, not necessarily
malicious activity, so many flags are normal admin work or AWS services.
Names and account numbers have been replaced with codes.
The HISTORY and ACCESS METHOD lines show what is normal for this identity; use them to judge
whether the activity fits its usual behaviour.

Using ONLY the evidence shown:
- verdict: malicious, suspicious, or benign
- root_cause: 1-2 sentences explaining what most likely happened and why it was flagged
- severity: low, medium, high, or critical
- confidence: a number from 0 to 1
- evidence: 2-5 short facts from the log that support your answer
- next_action: one concrete step an analyst should take next
If the evidence is not enough, say so in root_cause and lower your confidence.

"""

def cache_key(incident, variant):
    return f"{MODEL}|{variant}|{incident['arn_real']}|{incident['sourceIPAddress']}|{incident['hour']}"

def analyse(incident, df, variant="with_context"):
    key = cache_key(incident, variant)
    if key in CACHE:
        return CACHE[key], True
    t0 = time.perf_counter()
    context = build_context(incident, df)
    t1 = time.perf_counter()
    resp = client.models.generate_content(
        model=MODEL,
        contents=INSTRUCTIONS + context,
        config={"response_mime_type": "application/json", "response_schema": RCAResult, "temperature": 0},
    )
    t2 = time.perf_counter()
    entry = {
        "result": RCAResult.model_validate_json(resp.text).model_dump(),
        "context": context, "model": MODEL, "variant": variant,
        "context_seconds": round(t1 - t0, 3), "llm_seconds": round(t2 - t1, 3),
        "label_category": incident["label_category"],      # for scoring only, NOT sent to Gemini
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    CACHE[key] = entry
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(CACHE, f, indent=2)
    time.sleep(PAUSE_SECONDS)
    return entry, False

for ex in [ex1, ex2]:
    try:
        entry, cached = analyse(ex, df)
    except Exception as e:
        print("ERROR:", str(e)[:300])
        continue
    r = entry["result"]
    print("AI-SUGGESTED ROOT CAUSE (verify before acting)", "[from cache]" if cached else "[new Gemini call]")
    print(f"Identity: {ex['userIdentityarn']} | Hour: {ex['hour']} | True label (hidden from LLM): {ex['label_category']}")
    for k, v in r.items():
        print(f"  {k}: {v}")
    print(f"  Time: context {entry['context_seconds']} s + LLM {entry['llm_seconds']} s")
    print("=" * 80)

AI-SUGGESTED ROOT CAUSE (verify before acting) [from cache]
Identity: arn:aws:sts::123456789012:assumed-role/role-12/ec2-instance-41 | Hour: 2020-06-11 21:00:00+00:00 | True label (hidden from LLM): credential_abuse
  verdict: malicious
  root_cause: An automated discovery or enumeration script was run using compromised credentials from an EC2 instance role, resulting in thousands of AccessDenied errors across various AWS services.
  severity: high
  confidence: 0.9
  evidence: ['High volume of flagged events (428) and related events (3463) in a single hour', 'Extensive enumeration attempts across diverse AWS services such as IAM, S3, WAF, and Route53', '2974 AccessDenied errors and 369 Client.UnauthorizedOperation errors', 'Use of CLI and SDK scripts (Boto3 and AWS CLI) with unfamiliar IP activity']
  next_action: Revoke the active sessions for the compromised assumed-role and investigate the corresponding EC2 instance for potential malware or unauthorized access.
  Time: context 0.45

In [7]:
PLAN = {"credential_abuse": 3, "benign_owner_activity": 2, "benign_audit_tooling": 2,
        "benign_service": 1, "shared_cred_success": 1, "unlabelled": 1}

pool = real.drop(index=[ex1.name, ex2.name])           # keep ex1/ex2 out
practice = pd.concat([pool[pool["label_category"] == c].sample(n, random_state=7)
                      for c, n in PLAN.items()])
practice.to_parquet("practice_set.parquet")
print("Practice set saved:", len(practice), "incidents\n")

rows = []
for i, (_, incd) in enumerate(practice.iterrows(), 1):
    try:
        entry, cached = analyse(incd, df)
    except Exception as e:
        print(f"{i}. ERROR: {str(e)[:200]}")
        continue
    r = entry["result"]
    rows.append({"no": i, "true_label": incd["label_category"], "verdict": r["verdict"],
                 "severity": r["severity"], "confidence": r["confidence"],
                 "llm_s": entry["llm_seconds"], "root_cause": r["root_cause"]})
    print(f"{i}. [{incd['label_category']}] -> {r['verdict'].upper()} ({r['severity']}, conf {r['confidence']})")
    print(f"   {r['root_cause']}\n")

practice_results = pd.DataFrame(rows)
practice_results.to_csv("practice_results.csv", index=False)
print("Saved practice_results.csv")

Practice set saved: 10 incidents

1. [credential_abuse] -> SUSPICIOUS (low, conf 0.6)
   An established IAM role used an unfamiliar IP address and the AWS CLI to perform a ListBuckets request, which resulted in an AccessDenied error. This was flagged as an anomaly due to the new IP address usage combined with an authorization failure.

2. [credential_abuse] -> SUSPICIOUS (medium, conf 0.75)
   An established assumed role attempted to list S3 buckets from a new IP address, resulting in an AccessDenied error. This activity aligns with reconnaissance behavior by an unknown actor sharing an IP with another user.

3. [credential_abuse] -> SUSPICIOUS (medium, conf 0.6)
   An EC2 instance role performed enumeration actions (ListUsers and ListPolicies) from a new IP address, resulting in AccessDenied errors. This matches potential reconnaissance activity from a compromised instance or misconfigured script, though the evidence is limited.

4. [benign_owner_activity] -> BENIGN (low, conf 0.85)
 

In [8]:
try:
    lab = pd.read_csv("gold_set_to_label.csv", encoding="utf-8-sig")
except UnicodeDecodeError:
    lab = pd.read_csv("gold_set_to_label.csv", encoding="cp1252")
lab["true_verdict"] = lab["true_verdict"].astype(str).str.strip().str.lower()
lab["true_root_cause"] = lab["true_root_cause"].fillna("").astype(str).str.strip()

gold = pd.read_parquet("gold_set.parquet").drop(columns=["true_verdict", "true_root_cause"])
gold = gold.merge(lab[["gold_id", "true_verdict", "true_root_cause"]], on="gold_id", how="left")

bad_verdict = gold[~gold["true_verdict"].isin(["malicious", "suspicious", "benign"])]
empty_cause = gold[gold["true_root_cause"] == ""]
print("Gold incidents:", len(gold))
print("\nTrue verdicts:\n", gold["true_verdict"].value_counts().to_string())
print("\nProblems -> bad verdict:", list(bad_verdict["gold_id"]), "| empty root cause:", list(empty_cause["gold_id"]))
if bad_verdict.empty and empty_cause.empty:
    gold.to_parquet("gold_labelled.parquet", index=False)
    print("\nAll good. Saved gold_labelled.parquet")

Gold incidents: 44

True verdicts:
 true_verdict
benign        22
malicious     19
suspicious     3

Problems -> bad verdict: [] | empty root cause: []

All good. Saved gold_labelled.parquet


In [9]:
def build_minimal(incident):
    return (f"INCIDENT: identity={incident['userIdentityarn']} | ip={incident['sourceIPAddress']} | "
            f"hour={incident['hour']} | identity type={incident['identity_type']} | "
            f"flagged events={incident['n_events']} | most common event={incident['top_event']} | "
            f"error rate={incident['error_rate']:.0%}")

def analyse(incident, df, variant="with_context"):
    key = cache_key(incident, variant)
    if key in CACHE:
        return CACHE[key], True
    t0 = time.perf_counter()
    context = build_context(incident, df) if variant == "with_context" else build_minimal(incident)
    t1 = time.perf_counter()
    resp = client.models.generate_content(
        model=MODEL,
        contents=INSTRUCTIONS + context,
        config={"response_mime_type": "application/json", "response_schema": RCAResult, "temperature": 0},
    )
    t2 = time.perf_counter()
    entry = {
        "result": RCAResult.model_validate_json(resp.text).model_dump(),
        "context": context, "model": MODEL, "variant": variant,
        "context_seconds": round(t1 - t0, 3), "llm_seconds": round(t2 - t1, 3),
        "label_category": incident["label_category"],
        "created": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    CACHE[key] = entry
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(CACHE, f, indent=2)
    time.sleep(PAUSE_SECONDS)
    return entry, False

rows, total, done = [], len(gold) * 2, 0
for variant in ["with_context", "no_context"]:
    for _, g in gold.iterrows():
        done += 1
        try:
            entry, cached = analyse(g, df, variant)
        except Exception as e:
            print(f"[{done}/{total}] {g['gold_id']} {variant}: ERROR {str(e)[:120]}")
            continue
        r = entry["result"]
        rows.append({"gold_id": g["gold_id"], "variant": variant, "label_category": g["label_category"],
                     "true_verdict": g["true_verdict"], "llm_verdict": r["verdict"], "severity": r["severity"],
                     "confidence": r["confidence"], "context_seconds": entry["context_seconds"],
                     "llm_seconds": entry["llm_seconds"], "true_root_cause": g["true_root_cause"],
                     "llm_root_cause": r["root_cause"], "cached": cached})
        print(f"[{done}/{total}] {g['gold_id']} {variant}: {r['verdict']} (true: {g['true_verdict']})"
              + (" [cache]" if cached else ""))

results = pd.DataFrame(rows)
results.to_csv("gold_results.csv", index=False)
print("\nSaved gold_results.csv with", len(results), "of", total, "answers")

[1/88] G01 with_context: suspicious (true: malicious) [cache]
[2/88] G02 with_context: suspicious (true: malicious) [cache]
[3/88] G03 with_context: suspicious (true: malicious) [cache]
[4/88] G04 with_context: suspicious (true: malicious) [cache]
[5/88] G05 with_context: suspicious (true: malicious) [cache]
[6/88] G06 with_context: suspicious (true: malicious) [cache]
[7/88] G07 with_context: suspicious (true: malicious) [cache]
[8/88] G08 with_context: suspicious (true: malicious) [cache]
[9/88] G09 with_context: malicious (true: malicious) [cache]
[10/88] G10 with_context: benign (true: malicious) [cache]
[11/88] G11 with_context: suspicious (true: malicious) [cache]
[12/88] G12 with_context: suspicious (true: malicious) [cache]
[13/88] G13 with_context: suspicious (true: malicious) [cache]
[14/88] G14 with_context: suspicious (true: malicious) [cache]
[15/88] G15 with_context: suspicious (true: malicious) [cache]
[16/88] G16 with_context: benign (true: benign) [cache]
[17/88] G17 w

In [10]:
ALERT = {"malicious", "suspicious"}
res = results.copy()
res["exact"] = res["llm_verdict"] == res["true_verdict"]
res["alert_ok"] = res["llm_verdict"].isin(ALERT) == res["true_verdict"].isin(ALERT)
res["total_seconds"] = res["context_seconds"] + res["llm_seconds"]

print("=== SUMMARY ===")
print(res.groupby("variant").agg(
    n=("gold_id", "size"),
    exact_verdict_acc=("exact", "mean"),
    alert_acc=("alert_ok", "mean"),
    mean_confidence=("confidence", "mean"),
    mean_llm_s=("llm_seconds", "mean"),
    mean_total_s=("total_seconds", "mean"),
).round(3).to_string())

for v in ["with_context", "no_context"]:
    print(f"\n=== CONFUSION ({v}) rows = your answer, columns = Gemini ===")
    sub = res[res["variant"] == v]
    print(pd.crosstab(sub["true_verdict"], sub["llm_verdict"]).to_string())

print("\n=== CONFIDENCE when right vs wrong ===")
print(res.groupby(["variant", "exact"])["confidence"].mean().round(2).to_string())

print("\n=== ACCURACY BY CATEGORY (with_context) ===")
print(res[res["variant"] == "with_context"].groupby("label_category")["exact"].mean().round(2).to_string())

sheet = res[["gold_id", "variant", "true_verdict", "llm_verdict", "true_root_cause", "llm_root_cause"]].copy()
sheet["rc_score"] = ""
#sheet.sort_values(["gold_id", "variant"]).to_csv("root_cause_scoring.csv", index=False)
#print("\nSaved root_cause_scoring.csv")

=== SUMMARY ===
               n  exact_verdict_acc  alert_acc  mean_confidence  mean_llm_s  mean_total_s
variant                                                                                  
no_context    44              0.386      0.432            0.777       3.468         3.468
with_context  44              0.545      0.841            0.840       2.796         3.489

=== CONFUSION (with_context) rows = your answer, columns = Gemini ===
llm_verdict   benign  malicious  suspicious
true_verdict                               
benign            17          1           4
malicious          1          5          13
suspicious         1          0           2

=== CONFUSION (no_context) rows = your answer, columns = Gemini ===
llm_verdict   benign  suspicious
true_verdict                    
benign            15           7
malicious         17           2
suspicious         1           2

=== CONFIDENCE when right vs wrong ===
variant       exact
no_context    False    0.74
           

In [11]:
try:
    rc = pd.read_csv("root_cause_scoring.csv", encoding="utf-8-sig")
except UnicodeDecodeError:
    rc = pd.read_csv("root_cause_scoring.csv", encoding="cp1252")
rc["rc_score"] = pd.to_numeric(rc["rc_score"], errors="coerce")
bad = rc[~rc["rc_score"].isin([0, 1, 2])]
print("Rows:", len(rc), "| missing or invalid scores:", list(zip(bad["gold_id"], bad["variant"])))

rc = rc.merge(res[["gold_id", "variant", "label_category"]], on=["gold_id", "variant"], how="left")

print("\n=== ROOT CAUSE SCORE (0-2) ===")
print(rc.groupby("variant")["rc_score"].agg(mean="mean", n="count").round(2).to_string())

print("\n=== SCORE DISTRIBUTION (how many 0s, 1s, 2s) ===")
print(pd.crosstab(rc["variant"], rc["rc_score"]).to_string())

print("\n=== FULLY CORRECT (score 2), % ===")
print((rc.assign(full=rc["rc_score"] == 2).groupby("variant")["full"].mean() * 100).round(1).to_string())

print("\n=== MEAN SCORE BY CATEGORY (with_context) ===")
print(rc[rc["variant"] == "with_context"].groupby("label_category")["rc_score"].mean().round(2).to_string())

final = res.merge(rc[["gold_id", "variant", "rc_score"]], on=["gold_id", "variant"])
final.to_csv("final_evaluation.csv", index=False)
print("\nSaved final_evaluation.csv")

Rows: 88 | missing or invalid scores: []

=== ROOT CAUSE SCORE (0-2) ===
              mean   n
variant               
no_context    1.27  44
with_context  1.64  44

=== SCORE DISTRIBUTION (how many 0s, 1s, 2s) ===
rc_score      0   1   2
variant                
no_context    7  18  19
with_context  7   2  35

=== FULLY CORRECT (score 2), % ===
variant
no_context      43.2
with_context    79.5

=== MEAN SCORE BY CATEGORY (with_context) ===
label_category
benign_audit_tooling     1.67
benign_owner_activity    1.25
benign_service           2.00
credential_abuse         1.73
shared_cred_success      2.00
unlabelled               1.20

Saved final_evaluation.csv


In [12]:
ALERT = {"malicious", "suspicious"}
check = final[(final["llm_verdict"].isin(ALERT) != final["true_verdict"].isin(ALERT)) & (final["rc_score"] >= 1)]
print(len(check), "rows where Gemini's conclusion was wrong but the root cause got 1 or 2:\n")
for _, r in check.iterrows():
    print(f"{r['gold_id']} {r['variant']} | score {int(r['rc_score'])} | true: {r['true_verdict']} | Gemini: {r['llm_verdict']}")
    print(f"   Gemini said: {r['llm_root_cause'][:200]}\n")

18 rows where Gemini's conclusion was wrong but the root cause got 1 or 2:

G02 no_context | score 1 | true: malicious | Gemini: benign
   Gemini said: The flagged event was a ListBuckets call from an EC2 instance role that resulted in a 100% error rate. This anomaly flag likely represents a misconfigured application or a transient permissions issue 

G03 no_context | score 1 | true: malicious | Gemini: benign
   Gemini said: The identity attempted to list S3 buckets but encountered a 100 percent error rate, which likely triggered the anomaly detector due to the failure pattern rather than malicious intent.

G05 no_context | score 1 | true: malicious | Gemini: benign
   Gemini said: The flagged event was an authorization error for ListBuckets originating from an EC2 instance role. This is likely routine service startup activity or an application misconfiguration attempting to che

G06 no_context | score 1 | true: malicious | Gemini: benign
   Gemini said: The IAM role attempted to call

In [13]:
def read_csv_any(name):
    try:
        return pd.read_csv(name, encoding="utf-8-sig")
    except UnicodeDecodeError:
        return pd.read_csv(name, encoding="cp1252")

sc = read_csv_any("root_cause_scoring.csv")
sc["rc_score"] = pd.to_numeric(sc["rc_score"], errors="coerce")
ALERT = {"malicious", "suspicious"}
wrong = sc[sc["llm_verdict"].isin(ALERT) != sc["true_verdict"].isin(ALERT)]

print(len(wrong), "rows where Gemini's conclusion was wrong (highest allowed score = 1):\n")
for _, r in wrong.sort_values(["variant", "gold_id"]).iterrows():
    fix = "   <-- FIX: change to 1 (or 0)" if r["rc_score"] == 2 else ""
    print(f"{r['gold_id']:4} {r['variant']:13} true: {r['true_verdict']:10} Gemini: {r['llm_verdict']:10} score {int(r['rc_score'])}{fix}")

32 rows where Gemini's conclusion was wrong (highest allowed score = 1):

G01  no_context    true: malicious  Gemini: benign     score 0
G02  no_context    true: malicious  Gemini: benign     score 1
G03  no_context    true: malicious  Gemini: benign     score 1
G05  no_context    true: malicious  Gemini: benign     score 1
G06  no_context    true: malicious  Gemini: benign     score 1
G07  no_context    true: malicious  Gemini: benign     score 1
G08  no_context    true: malicious  Gemini: benign     score 0
G09  no_context    true: malicious  Gemini: benign     score 1
G11  no_context    true: malicious  Gemini: benign     score 1
G12  no_context    true: malicious  Gemini: benign     score 0
G13  no_context    true: malicious  Gemini: benign     score 0
G14  no_context    true: malicious  Gemini: benign     score 1
G15  no_context    true: malicious  Gemini: benign     score 0
G16  no_context    true: benign     Gemini: suspicious score 1
G17  no_context    true: benign     Gemini: 